## MNIST → HLS4ML (QONNX flow)

Converts `mnist_finn_ready.onnx` (Brevitas QAT export from `train_qat.ipynb`) to a
Vivado HLS project via hls4ml — **keeping the Quant nodes**, so the trained
bit-widths and scale factors survive into hardware.

Why this matters: the old version of this notebook *stripped* every `Quant` node
and forced a blanket `ap_fixed<8,4>` on the whole network. That threw away all the
QAT information and destroyed accuracy (see the notes at the bottom). This version
uses hls4ml's native QONNX ingestion, which turns each Quant node into the correct
fixed-point type per layer, and validates the result against MNIST in C-simulation.

In [1]:
import os
import shutil

import numpy as np
import hls4ml

# hls4ml 1.3.0 bug: InferPrecisionTypes.match() calls self.inputs[0] unconditionally,
# but Constant layers (weights) always have inputs=[]. Patch it to also match
# constants so _infer_default_type() resolves their UnspecifiedPrecisionType.
import hls4ml.model.optimizer.passes.infer_precision as _ip
from hls4ml.model.types import UnspecifiedPrecisionType as _UPT

_orig_ip_match = _ip.InferPrecisionTypes.match

def _safe_ip_match(self, node):
    if not node.inputs:
        return any(isinstance(lt.precision, _UPT) for lt in node.types.values())
    return _orig_ip_match(self, node)

_ip.InferPrecisionTypes.match = _safe_ip_match

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup_model
from qonnx.transformation.channels_last import ConvertToChannelsLastAndClean
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from qonnx.transformation.general import SortGraph

print('Imports OK')

Imports OK


### Prepare the QONNX model

Standard qonnx cleanup + channels-last + Gemm→MatMul, plus one custom rewrite:

**Move ReLU Quant nodes past MaxPool** (`Relu → Quant → MaxPool` becomes
`Relu → MaxPool → Quant`). This is numerically exact — max-pooling commutes with a
monotonic elementwise quantizer. It matters because hls4ml folds a conv's quantized
weights only when the data-path scale sits directly on the conv input; with the
Quant stuck above the pool, conv2 was left as an unfused generic `Conv` layer.

In [2]:
def move_quant_past_maxpool(model):
    """Rewrite Quant -> MaxPool into MaxPool -> Quant (exact: max-pooling
    commutes with monotonic elementwise quantization)."""
    graph = model.graph
    consumers = {}
    for n in graph.node:
        for i in n.input:
            consumers.setdefault(i, []).append(n)
    changed = False
    for quant in list(graph.node):
        if quant.op_type != 'Quant':
            continue
        cons = consumers.get(quant.output[0], [])
        if len(cons) != 1 or cons[0].op_type != 'MaxPool':
            continue
        pool = cons[0]
        q_in, q_out, p_out = quant.input[0], quant.output[0], pool.output[0]
        mid = q_out + '_prepool'
        pool.input[0] = q_in
        pool.output[0] = mid
        quant.input[0] = mid
        quant.output[0] = p_out
        changed = True
    if changed:
        model = model.transform(SortGraph())
        model = cleanup_model(model)
    return model


qonnx_model = ModelWrapper('mnist_finn_ready.onnx')
qonnx_model = cleanup_model(qonnx_model)
qonnx_model = qonnx_model.transform(ConvertToChannelsLastAndClean(make_input_channels_last=True))
qonnx_model = qonnx_model.transform(GemmToMatMul())
qonnx_model = cleanup_model(qonnx_model)
qonnx_model = move_quant_past_maxpool(qonnx_model)

print('Model ready. Ops:', [n.op_type for n in qonnx_model.graph.node])

Model ready. Ops: ['Quant', 'Quant', 'Quant', 'Quant', 'Quant', 'Quant', 'Quant', 'Quant', 'Quant', 'Conv', 'Relu', 'MaxPool', 'Quant', 'Conv', 'Relu', 'MaxPool', 'Quant', 'Flatten', 'MatMul', 'Add', 'Relu', 'Quant', 'MatMul', 'Add']


/Users/dhruv.ramaswamy/Documents/MNIST_MODEL/.venv/lib/python3.12/site-packages/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 17. Thus the transformation may fail or return incomplete results.
  warnings.warn(


### Generate hls4ml config

`config_from_onnx_model` reads the Quant nodes and sets per-layer precisions itself
(4/2-bit weights, 2-bit activations, 8-bit input). The default precision only covers
the *glue* between quantized points — the scale/rescale (ApplyAlpha) stages and ReLU
buffers. It must be wide enough to hold `1/input_scale ≈ 45` and the small bias
scales (~0.003): `ap_fixed<24,12>` covers both (`ap_fixed<32,16>` gains only ~0.1%).

Do **not** set `config['Model']['Precision']` afterwards — that was the old bug.

In [3]:
config = hls4ml.utils.config_from_onnx_model(
    qonnx_model, granularity='name', backend='Vivado',
    default_precision='ap_fixed<24,12>')

config['Model']['ReuseFactor'] = 32
config['Model']['Strategy']    = 'Resource'

# Propagate model-level ReuseFactor to all layers so per-layer defaults don't override it
for layer in config.get('LayerName', {}).values():
    layer['ReuseFactor'] = 32

print('Config ready. Layers:')
for name in config.get('LayerName', {}):
    print(' ', name)

Output layers:  ['Add_1']
Input shape: [28, 28, 1]
Topology:
Layer name: Quant_0, layer type: Quant, current shape: [[16, 3, 3, 1]]
Layer name: Quant_1, layer type: Quant, current shape: [[16], [1]]
Layer name: Quant_2, layer type: Quant, current shape: [[16, 3, 3, 16]]
Layer name: Quant_3, layer type: Quant, current shape: [[16], [1]]
Layer name: Quant_4, layer type: Quant, current shape: [[144, 64]]
Layer name: Quant_5, layer type: Quant, current shape: [[64], [1]]
Layer name: Quant_6, layer type: Quant, current shape: [[64, 10]]
Layer name: Quant_7, layer type: Quant, current shape: [[10], [1]]
Layer name: Quant_8, layer type: Quant, current shape: [[1, 28, 28, 1]]
Layer name: Conv_0, layer type: Conv, current shape: [[1, 28, 28, 1], [16, 3, 3, 1], [16]]
Layer name: Relu_0, layer type: Activation, current shape: [[1, 28, 28, 16]]
Layer name: MaxPool_0, layer type: MaxPooling2D, current shape: [[1, 28, 28, 16]]
Layer name: Quant_9, layer type: Quant, current shape: [[1, 14, 14, 16]]


### Convert and write HLS project

In [4]:
if os.path.exists('my_hls_project'):
    shutil.rmtree('my_hls_project')

hls_model = hls4ml.converters.convert_from_onnx_model(
    qonnx_model,
    hls_config=config,
    output_dir='my_hls_project',
    backend='Vivado',
    part='xc7z020clg484-1',
    clock_period=10,
    io_type='io_stream',
)
hls_model.write()

print('HLS project written to: my_hls_project/')
print('Contents:', sorted(os.listdir('my_hls_project')))

Interpreting Model ...
Output layers:  ['Add_1']
Input shape: [28, 28, 1]
Topology:
Layer name: Quant_0, layer type: Quant, current shape: [[16, 3, 3, 1]]
Layer name: Quant_1, layer type: Quant, current shape: [[16], [1]]
Layer name: Quant_2, layer type: Quant, current shape: [[16, 3, 3, 16]]
Layer name: Quant_3, layer type: Quant, current shape: [[16], [1]]
Layer name: Quant_4, layer type: Quant, current shape: [[144, 64]]
Layer name: Quant_5, layer type: Quant, current shape: [[64], [1]]
Layer name: Quant_6, layer type: Quant, current shape: [[64, 10]]
Layer name: Quant_7, layer type: Quant, current shape: [[10], [1]]
Layer name: Quant_8, layer type: Quant, current shape: [[1, 28, 28, 1]]
Layer name: Conv_0, layer type: Conv, current shape: [[1, 28, 28, 1], [16, 3, 3, 1], [16]]
Layer name: Relu_0, layer type: Activation, current shape: [[1, 28, 28, 16]]
Layer name: MaxPool_0, layer type: MaxPooling2D, current shape: [[1, 28, 28, 16]]
Layer name: Quant_9, layer type: Quant, current sh

/Users/dhruv.ramaswamy/Documents/MNIST_MODEL/.venv/lib/python3.12/site-packages/hls4ml/model/optimizer/passes/move_scales.py:183: UserWarning: Failed to propagate quantization bias down Add node; model probably not suppored.
  warnings.warn('Failed to propagate quantization bias down Add node; model probably not suppored.', stacklevel=1)


### Validate accuracy in C-simulation

Compiles the generated C++ (no Vivado needed) and runs the MNIST test set through
it. `patch_ap_types` fixes a macOS/libc++ clash in Xilinx's bundled headers
(`ap_*_special.h` forward-declare `complex` in `namespace std`, which is ambiguous
with libc++'s versioned `std::__1::complex`) — it only touches the copies inside
`my_hls_project/`, and is a no-op on Linux/Vivado.

Reference: the Brevitas model from `train_qat.ipynb` scores **97.3%** on the first
1000 test images (97.9% on all 10k). The HLS model should land within ~0.5% of that.

In [5]:
def patch_ap_types(project_dir):
    old = ("// #include <complex>\n"
           "namespace std {\n"
           "template<typename _Tp> class complex;\n"
           "}\n")
    for name in ('ap_int_special.h', 'ap_fixed_special.h'):
        path = os.path.join(project_dir, 'firmware', 'ap_types', name)
        with open(path) as f:
            src = f.read()
        if old in src:
            with open(path, 'w') as f:
                f.write(src.replace(old, '#include <complex>\n'))

patch_ap_types('my_hls_project')
hls_model._compile()   # not .compile(): that re-writes the project and undoes the patch

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081]),
])
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=data_transforms)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

images, labels = next(iter(test_loader))
x = np.ascontiguousarray(images.numpy().transpose(0, 2, 3, 1))  # NCHW -> NHWC
y = hls_model.predict(x)
pred = y.reshape(len(labels), 10).argmax(axis=1)
acc = (pred == labels.numpy()).mean()
print(f'HLS csim accuracy on {len(labels)} test images: {100 * acc:.2f}%')

HLS csim accuracy on 1000 test images: 96.90%


### Synthesize (requires Vivado HLS on PATH)

In [6]:
# Full synthesis:
# hls_model.build(csim=False, synth=True, export=True)

---
### What was wrong before (post-mortem)

The old notebook produced near-random accuracy because of two compounding mistakes:

1. **It stripped every `Quant` node** before handing the graph to hls4ml. All the
   QAT scale factors and bit-widths — the entire point of training with Brevitas —
   were discarded, so hls4ml saw a plain float model.
2. **It then forced `ap_fixed<8,4>` on the whole network** (range −8…+7.94,
   resolution 1/16). Against the actual trained values this fails in both directions:
   - fc2 weights live in ±0.18 with a quant step of 0.026 → almost all collapse to
     0 or ±0.0625; conv1 biases (scale 0.003) all round to exactly 0.
   - relu2/relu3 are 2-bit activations with scales 3.70/3.38, so legal outputs reach
     ~11.1 → **overflows** ap_fixed<8,4>'s max of 7.94 and *wraps to negative*.

Bonus bug: its saved outputs show 32/64/128-wide layers — it had last been run
against a stale ONNX export of the old, bigger architecture.

Two more hls4ml 1.3.0 quirks the new flow works around:
- With non-power-of-2 scales, a Quant becomes scale → clip/round → rescale
  (ApplyAlpha pairs). The scale constants take the *model default* precision, so the
  default must fit `1/0.0223 ≈ 44.8` — the original `ap_fixed<8,4>`/`<16,6>` wrapped it.
- Conv weight folding (`ScaleDownConv`) requires the data-path scale adjacent to the
  conv; the `move_quant_past_maxpool` rewrite above makes that true for conv2.